[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fedgutcor/lab-ia-intermedio/blob/main/recetas/03_transferencia_y_lora.ipynb)

# 03 · Transferencia y LoRA: adaptar sin empezar de cero

**M3 · Ajuste fino y adaptación de modelos** · Laboratorio de IA · nivel intermedio

En esta receta vas a:
- reutilizar una red ya entrenada (MobileNetV2) con la base congelada
- entrenar solo una cabeza nueva con tus categorías
- adaptar con LoRA: pocos parámetros entrenables y pesos originales intactos
- comparar antes y después con el mismo protocolo

**Cómo usarla:** en Colab elige *Entorno de ejecución → Ejecutar todo*. No hace falta instalar nada. Si Colab avisa que el notebook no es de Google, elige *Ejecutar de todos modos*. Cuando termine, busca la celda **«Cambia esto»**, modifica un valor y vuelve a ejecutar.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path
import tensorflow as tf
import keras
from keras import layers

INICIO = time.time()
keras.utils.set_random_seed(0)

## 1 · Los datos

**flower_photos** de TensorFlow: 3.670 fotos de 5 tipos de flores. Licencia CC BY 2.0; la lista de fotógrafos viene en el `LICENSE.txt` del archivo. Para que sea rápido en CPU usamos tres porciones pequeñas que no se mezclan:

- **inicial:** 768 fotos para entrenar la cabeza;
- **nuevos:** otras 768 que llegan después, para adaptar con LoRA;
- **validación:** 384 que no se usan para entrenar nada.

In [ ]:
URL_FLORES = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archivo = Path(keras.utils.get_file("flower_photos.tgz", origin=URL_FLORES))
carpeta = archivo.parent / "flower_photos"
if not carpeta.exists():
    with tarfile.open(archivo) as t:
        t.extractall(archivo.parent, filter="data")
print((carpeta / "LICENSE.txt").read_text().splitlines()[0])

TAM = 160
CLASES = sorted(d.name for d in carpeta.iterdir() if d.is_dir())
rutas = sorted(carpeta.glob("*/*.jpg"))
np.random.default_rng(0).shuffle(rutas)          # se mezcla UNA vez: ninguna foto queda en dos porciones

def porcion(lista):
    def cargar(ruta, y):
        img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
        return tf.image.resize(img, (TAM, TAM)), y
    ys = [CLASES.index(p.parent.name) for p in lista]
    return tf.data.Dataset.from_tensor_slices(([str(p) for p in lista], ys)).map(cargar).batch(64)

inicial, nuevos, validacion = porcion(rutas[:768]), porcion(rutas[768:1536]), porcion(rutas[1536:1920])
print("Clases:", CLASES, "· fotos en total:", len(rutas))

imgs, etiquetas = next(iter(inicial))
fig, ejes = plt.subplots(1, 6, figsize=(12, 2.2))
for eje, img, e in zip(ejes, imgs, etiquetas):
    eje.imshow(img.numpy().astype("uint8")); eje.set_title(CLASES[int(e)], fontsize=9); eje.axis("off")
plt.show()

## 2 · La base congelada

MobileNetV2 ya aprendió a reconocer bordes, texturas y formas con millones de imágenes. La usamos **sin cambiar sus pesos** para convertir cada foto en 1.280 números (sus características). Este paso es el más lento de la receta.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(TAM, TAM, 3), include_top=False, pooling="avg", weights="imagenet")
base.trainable = False

def caracteristicas(ds):
    X, Y = [], []
    for imgs, ys in ds:
        X.append(base(keras.applications.mobilenet_v2.preprocess_input(imgs), training=False).numpy())
        Y.append(ys.numpy())
    return np.concatenate(X), np.concatenate(Y)

X_ini, y_ini = caracteristicas(inicial)
X_nue, y_nue = caracteristicas(nuevos)
X_val, y_val = caracteristicas(validacion)
print("Características:", X_ini.shape, X_nue.shape, X_val.shape, f"· {time.time() - INICIO:.0f} s")

## 3 · Cambia esto

In [ ]:
RANGO_LORA = 4      # rango de los adaptadores LoRA (prueba 1 o 16)
EPOCAS = 15

## 4 · Transferencia: una cabeza nueva sobre la base

In [ ]:
cabeza = keras.Sequential([
    keras.Input((1280,)),
    layers.Dense(128, activation="relu", name="oculta"),
    layers.Dense(len(CLASES), activation="softmax", name="salida"),
])
cabeza.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cabeza.fit(X_ini, y_ini, epochs=EPOCAS, batch_size=32, verbose=0)
exactitud_inicial = cabeza.evaluate(X_val, y_val, verbose=0)[1]

def contar(modelo):
    entrenables = sum(int(np.prod(v.shape)) for v in modelo.trainable_weights)
    congelados = sum(int(np.prod(v.shape)) for v in modelo.non_trainable_weights)
    return entrenables, congelados

print(f"Exactitud en validación con la cabeza inicial: {exactitud_inicial:.3f}")
print(f"Base MobileNetV2 congelada: {base.count_params():,} parámetros que NO se entrenan")
print(f"Cabeza: {contar(cabeza)[0]:,} parámetros entrenables")

## 5 · LoRA: adaptar con los datos nuevos

En vez de reentrenar todos los pesos de la cabeza, **LoRA** los congela y agrega a cada capa dos matrices pequeñas (de rango bajo) que sí se entrenan. Los pesos originales no cambian: el adaptador se suma encima.

In [ ]:
oculta = cabeza.get_layer("oculta")
pesos_originales = oculta.kernel.numpy().copy()
for nombre in ["oculta", "salida"]:
    cabeza.get_layer(nombre).enable_lora(RANGO_LORA)

entrenables, congelados = contar(cabeza)
print(f"Con LoRA (rango {RANGO_LORA}): {entrenables:,} entrenables · {congelados:,} congelados")

cabeza.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cabeza.fit(X_nue, y_nue, epochs=EPOCAS, batch_size=32, verbose=0)
exactitud_lora = cabeza.evaluate(X_val, y_val, verbose=0)[1]

# Con LoRA activo, `kernel` devuelve el peso EFECTIVO (original + adaptador);
# `_kernel` es la variable original, congelada.
intactos = np.allclose(pesos_originales, oculta._kernel.numpy())
cambio_efectivo = float(np.abs(oculta.kernel.numpy() - pesos_originales).mean())
print(f"Exactitud en validación: inicial {exactitud_inicial:.3f} → con LoRA {exactitud_lora:.3f}")
print("¿Los pesos originales siguen intactos?", intactos)
print(f"Cambio medio del peso efectivo gracias al adaptador: {cambio_efectivo:.5f}")
print(f"Tiempo total de la receta: {time.time() - INICIO:.0f} s")

## Cómo leer el resultado

- La validación es pequeña: una diferencia de pocos puntos puede ser ruido. Es una **exploración**, no una conclusión general.
- Si mejora, puede deberse a que hubo **más datos**, no a LoRA en sí. Para atribuir el efecto al método habría que comparar contra otras opciones con los mismos datos.
- En una red de este tamaño LoRA ahorra poco. Su ventaja real aparece en modelos con millones de parámetros por capa.
- **Cargar un adaptador hecho por otra persona no es entrenarlo.** El entrenamiento propio se demuestra con datos, configuración y resultados registrados.